In [ ]:
import mne
import random
import utils
import os
import pickle
import algo
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib widget

In [ ]:
def prepare_data_subj(Subj_ID, fs):
    eeg_list, eog_list, gaze_list, feats_list = utils.load_subj(Subj_ID)
    objflow_list = [np.expand_dims(feats[:,8,:], axis=1) for feats in feats_list]
    eeg_reg_list = [utils.regress_out(eeg, eog) for eeg, eog in zip(eeg_list, eog_list)]
    eeg_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in eeg_list]
    eog_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in eog_list]
    gaze_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in gaze_list]
    gaze_velocity_list = [utils.calcu_gaze_velocity(gaze) for gaze in gaze_list]
    objflow_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in objflow_list]
    eeg_reg_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in eeg_reg_list]
    data_multitask_dict = {'EEG': eeg_list, 'EOG': eog_list, 'GAZE': gaze_list, 'GAZE_V': gaze_velocity_list, 'EEG-EOG': eeg_reg_list}
    return data_multitask_dict, objflow_list

## Load data

In [ ]:
subjects = ['Subj_1', 'Subj_2', 'Subj_3', 'Subj_4', 'Subj_5', 'Subj_6', 'Subj_7', 'Subj_8', 'Subj_9']
subj_path = ['../../Experiments/data/SOMove_MultiTask/' + sub + '/' for sub in subjects]
nb_subj = len(subjects)
bads = [[], ['A6','A24','B14','B25'], ['B25','B26'], ['A2','A8','A9','A15','B25','B26','B27','B31'], ['B25','B31'], ['B12'], ['B19','A30'], ['A5','A9','B5','B25','B29'], ['A11','B4','B25','B29']] # bad channels
fs = 30
feats_path_folder = '../Feat_Multi/features/'
MODs = ['EEG', 'EOG', 'GAZE', 'GAZE_V', 'EEG-EOG']
# create table and figure folders for each modality
for mod in MODs:
    table_folder = 'tables/' + mod + '/'
    fig_folder = 'figures/' + mod + '/'
    if not os.path.exists(table_folder):
        os.makedirs(table_folder)
    if not os.path.exists(fig_folder):
        os.makedirs(fig_folder)

In [ ]:
%%capture
LOAD_ONLY = True
ALL_NEW = False
len_video = 180
if not LOAD_ONLY:
    for TASK in ['1', '2', '3']:
        if ALL_NEW:
            _, _, _, _, _ = utils.data_multi_subj(subj_path, fs, bads, feats_path_folder, len_video, SAVE=True, Task=TASK)
        else:
            _, _, _, _, _ = utils.add_new_data(subj_path, fs, bads, feats_path_folder, len_video, Task=TASK)

In [ ]:
L_EEG = 3
L_Stim = 15
offset_EEG = 1
offset_Stim = 0

RUN_TIMES = 10
MOD = 'EEG-EOG'
trial_len = 30
BOOTSTRAP = True
n_components = 5
range_into_account = 3
nb_comp_into_account = 2

## CCA

In [ ]:
def analyze_all(fs, L_EEG, L_Stim, offset_EEG, offset_Stim, range_into_account, nb_comp_into_account, trial_len=30, n_components=5, save_name=None, MOD='EEG-EOG', PERMU_TEST=False, BOOTSTRAP=True):
    all_acc = []
    corr_match_all_subj = []
    corr_mismatch_all_subj = []
    acc_permu_all = []
    start_points = None
    for Subj_ID in range(len(subjects)):
        print(f"###################\nSubject {Subj_ID + 1} / {len(subjects)}")
        data_multitask_dict, objflow_list = prepare_data_subj(Subj_ID, fs)
        CCA = algo.CanonicalCorrelationAnalysis(data_multitask_dict[MOD], objflow_list, fs, L_EEG, L_Stim, offset_EEG, offset_Stim, leave_out=1, n_components=n_components)
        corr_match_data, corr_mismatch_data, acc_permu_list, start_points = CCA.match_mismatch(trial_len=trial_len, PERMU_TEST=PERMU_TEST, BOOTSTRAP=BOOTSTRAP, given_start_points=start_points)
        print("###########Match-Mismatch, TASK 1, 2, 3###########")
        acc_all_tasks, _, _, _, _ = utils.eval_compete_3D(corr_match_data, corr_mismatch_data, True, range_into_account=range_into_account, nb_comp_into_account=nb_comp_into_account, message=True)
        print("###########TASK 2 vs TASK 1###########")
        acc_2vs1, _, _, _, _ = utils.eval_compete(corr_match_data[:,:,1], corr_match_data[:,:,0], True, range_into_account=range_into_account, nb_comp_into_account=nb_comp_into_account, message=True)
        print("###########TASK 3 vs TASK 2###########")
        acc_3vs2, _, _, _, _ = utils.eval_compete(corr_match_data[:,:,2], corr_match_data[:,:,1], True, range_into_account=range_into_account, nb_comp_into_account=nb_comp_into_account, message=True)
        all_acc.append({
            'Subject': Subj_ID + 1,
            'Task_1': acc_all_tasks[0],
            'Task_2': acc_all_tasks[1],
            'Task_3': acc_all_tasks[2],
            'Task_2vs1': acc_2vs1,
            'Task_3vs2': acc_3vs2,
        })
        corr_match_all_subj.append(corr_match_data)
        corr_mismatch_all_subj.append(corr_mismatch_data)
        if PERMU_TEST:
            acc_permu_all += acc_permu_list
    all_acc = pd.DataFrame(all_acc)
    if PERMU_TEST:
        acc_permu = np.concatenate(acc_permu_all, axis=0)
        alpha = 0.05
        lower_bound = np.percentile(acc_permu, alpha/2*100)
        upper_bound = np.percentile(acc_permu, (1-alpha/2)*100)
    else:
        lower_bound = None
        upper_bound = None
    # add two columns to all_acc for lower and upper bound
    all_acc['lower_bound'] = lower_bound
    all_acc['upper_bound'] = upper_bound
    if save_name is not None:
        save_path = f"tables/{MOD}/{save_name}"
        all_acc.to_csv(f"{save_path}_acc_{trial_len}{'_BT' if BOOTSTRAP else ''}.csv", index=False)
        # save corr_match_all_subj, corr_mismatch_all_subj, start_idx as dictionary
        corr_res = {
            'corr_match_all_subj': corr_match_all_subj,
            'corr_mismatch_all_subj': corr_mismatch_all_subj,
            'start_points': start_points
        }
        # save as pickle file
        with open(f"{save_path}_corr_{trial_len}{'_BT' if BOOTSTRAP else ''}.pickle", 'wb') as f:
            pickle.dump(corr_res, f)
    return all_acc, corr_match_all_subj, corr_mismatch_all_subj, start_points

In [ ]:
# create a dictionary to store the results
for i in range(RUN_TIMES):
    save_name = f"RUN_{i+1}"
    PERMU_TEST = i == 0
    all_acc, corr_match_all_subj, corr_mismatch_all_subj, start_idx = analyze_all(fs, L_EEG, L_Stim, offset_EEG, offset_Stim, range_into_account, nb_comp_into_account, n_components=n_components, save_name=save_name, MOD=MOD, trial_len=trial_len, PERMU_TEST=PERMU_TEST, BOOTSTRAP=BOOTSTRAP)
    print(all_acc)

In [ ]:
# load the results
all_acc = []
for i in range(RUN_TIMES):
    save_name = f"RUN_{i+1}"
    save_path = f"tables/{MOD}/{save_name}"
    all_acc.append(pd.read_csv(f"{save_path}_acc_{trial_len}{'_BT' if BOOTSTRAP else ''}.csv"))
# average the results
all_acc = pd.concat(all_acc, ignore_index=True)
all_acc = all_acc.groupby(['Subject']).mean().reset_index()

In [ ]:
plt.figure(figsize=(5, 4))

# Create colormap for different subjects
n_subjects = len(all_acc)
colors = plt.cm.rainbow(np.linspace(0, 1, n_subjects))

# Plot individual subject lines with different colors
for idx, (row, color) in enumerate(zip(all_acc.iterrows(), colors)):
    plt.plot([1, 2, 3], 
            [row[1]['Task_1'], row[1]['Task_2'], row[1]['Task_3']], 
            'o-', alpha=0.7, color=color, label=f'Subject {int(row[1]["Subject"])}')
# Add horizontal line at significance levels (0.54 and 0.45)
plt.axhline(y=all_acc['upper_bound'].mean(), color='grey', linestyle='--', alpha=0.8)
plt.axhline(y=all_acc['lower_bound'].mean(), color='grey', linestyle='--', alpha=0.8)
# Customize plot
plt.xticks([1, 2, 3], 
          ['Task_1', 'Task_2', 'Task_3'], 
          rotation=45)
plt.ylabel('Accuracy')
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Set style
plt.figure(figsize=(4, 4))
# Create box plot
# sns.boxplot(data=all_acc[['Task_1', 'Task_2', 'Task_3', 'Task_2vs1', 'Task_3vs2']])
sns.boxplot(data=all_acc[['Task_1', 'Task_2', 'Task_3']])
# Customize plot
plt.title('Accuracy of match-mismatch task', fontsize=12)
plt.ylabel('Accuracy', fontsize=10)
plt.xlabel('Tasks', fontsize=10)
# Add horizontal line at significance levels (0.54 and 0.45)
plt.axhline(y=all_acc['upper_bound'].mean(), color='grey', linestyle='--', alpha=0.8)
plt.axhline(y=all_acc['lower_bound'].mean(), color='grey', linestyle='--', alpha=0.8)
# Rotate x-axis labels for better readability
plt.xticks(rotation=45)
# Add individual points
# sns.swarmplot(data=all_acc[['Task_1', 'Task_2', 'Task_3', 'Task_2vs1', 'Task_3vs2']], color='black', alpha=0.5, size=4)
sns.swarmplot(data=all_acc[['Task_1', 'Task_2', 'Task_3']], color='black', alpha=0.5, size=4)
# plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(4, 4))
# Create box plot
sns.boxplot(data=all_acc[['Task_2vs1', 'Task_3vs2']])
# Customize plot
plt.title('Accuracy of competing task', fontsize=12)
plt.ylabel('Accuracy', fontsize=10)
plt.xlabel('Tasks', fontsize=10)
# Add horizontal line at significance levels (0.54 and 0.45)
plt.axhline(y=all_acc['upper_bound'].mean(), color='grey', linestyle='--', alpha=0.8)
plt.axhline(y=all_acc['lower_bound'].mean(), color='grey', linestyle='--', alpha=0.8)
# Rotate x-axis labels for better readability
plt.xticks(rotation=45)
# Add individual points
sns.swarmplot(data=all_acc[['Task_2vs1', 'Task_3vs2']], color='black', alpha=0.5, size=4)
# plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def acc_fine_grained(corr_match_eeg, corr_mismatch_eeg, nb_blocks, nb_folds=7, range_into_account=3, nb_comp_into_account=2):
    corr_m_folds = np.array_split(corr_match_eeg, nb_folds, axis=0)
    corr_mm_folds = np.array_split(corr_mismatch_eeg, nb_folds, axis=0)
    corr_m_blocks = [np.array_split(corr_m, nb_blocks, axis=0) for corr_m in corr_m_folds]
    corr_mm_blocks = [np.array_split(corr_mm, nb_blocks, axis=0) for corr_mm in corr_mm_folds]
    block_res_folds = []
    for i in range(nb_folds):
        block_res = [utils.eval_compete_3D(m, mm, True, range_into_account=range_into_account, nb_comp_into_account=nb_comp_into_account, message=False)[0] for m, mm in zip(corr_m_blocks[i], corr_mm_blocks[i])]
        block_res = np.stack(block_res, axis=0)
        block_res_folds.append(block_res)
    block_res_avg = np.mean(block_res_folds, axis=0)
    return block_res_avg

In [ ]:
block_res_avg_all_runs = []
for i in range(RUN_TIMES):
    save_name = f"RUN_{i+1}"
    save_path = f"tables/EEG-EOG/{save_name}"
    with open(f"{save_path}_corr_{trial_len}{'_BT' if BOOTSTRAP else ''}.pickle", 'rb') as f:
        corr_res = pickle.load(f)
    corr_match_all_subj = corr_res['corr_match_all_subj']
    corr_mismatch_all_subj = corr_res['corr_mismatch_all_subj']
    block_res_all_subj = []
    for corr_match_eeg, corr_mismatch_eeg in zip(corr_match_all_subj, corr_mismatch_all_subj):
        block_res_avg = acc_fine_grained(corr_match_eeg, corr_mismatch_eeg, nb_blocks=10, nb_folds=7, range_into_account=range_into_account, nb_comp_into_account=nb_comp_into_account)
        block_res_all_subj.append(block_res_avg)
    block_res_avg_all_runs.append(np.mean(block_res_all_subj, axis=0))
block_res_avg_all_runs = np.mean(block_res_avg_all_runs, axis=0)

In [ ]:
# block_res_avg_all_runs
# plot the results
plt.figure(figsize=(8, 4))
plt.plot(block_res_avg_all_runs[:,0], 'o-', label='Task 1', alpha=0.7, color='blue')
plt.plot(block_res_avg_all_runs[:,1], 'o-', label='Task 2', alpha=0.7, color='orange')
plt.plot(block_res_avg_all_runs[:,2], 'o-', label='Task 3', alpha=0.7, color='green')
plt.xticks([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [f'{i+1}' for i in range(10)])
plt.ylabel('Accuracy')
plt.xlabel('Block Number')
plt.title('Accuracy per Time Block (averaged over all videos, subjects and runs)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def process_corr(corr_tensor, range_into_account=3, nb_comp_into_account=2):
    _, _, nb_tasks = corr_tensor.shape
    corr_tensor = corr_tensor[:,:range_into_account,:]
    for i in range(nb_tasks):
        # sort the components for each task from the most to the least correlated
        corr_tensor[:, :, i] = np.sort(corr_tensor[:, :, i], axis=1)[:, ::-1]
    # take the sum of the first nb_comp_into_account components
    corr_tensor = np.sum(corr_tensor[:, :nb_comp_into_account, :], axis=1) # shape: (nb_trials, nb_tasks)
    return corr_tensor


In [ ]:
save_path = f"tables/{MOD}/RUN_1_corr_{trial_len}{'_BT' if BOOTSTRAP else ''}.pickle"
with open(save_path, 'rb') as f:
    corr_res = pickle.load(f)
corr_match_all_subj = corr_res['corr_match_all_subj']
corr_mismatch_all_subj = corr_res['corr_mismatch_all_subj']
start_points = corr_res['start_points']

In [ ]:
corr_match_processed = [process_corr(corr_match) for corr_match in corr_match_all_subj]
corr_mismatch_processed = [process_corr(corr_mismatch) for corr_mismatch in corr_mismatch_all_subj]
corr_match_avg_subj = np.mean(corr_match_processed, axis=0)
corr_mismatch_avg_subj = np.mean(corr_mismatch_processed, axis=0)

In [ ]:
plt.close('all')
corr_match_folds = np.split(corr_match_avg_subj, 7)
corr_match_avg = np.mean(corr_match_folds, axis=0)
plt.figure(figsize=(8, 4))
plt.plot(start_points/fs, corr_match_avg[:,0], label='TASK 1')
plt.plot(start_points/fs, corr_match_avg[:,1], label='TASK 2')
plt.plot(start_points/fs, corr_match_avg[:,2], label='TASK 3')
plt.legend()
plt.show()
plt.xlabel('Start Time (s)')
plt.ylabel('CC1+CC2')

In [ ]:
# plot for all subjects
fig, axes = plt.subplots(3, 3, figsize=(12, 8), sharex=True, sharey=True)
axes = axes.flatten()
for i, (corr_match, corr_mismatch) in enumerate(zip(corr_match_processed, corr_mismatch_processed)):
    corr_match_folds = np.split(corr_match, 7)
    corr_match_avg = np.mean(corr_match_folds, axis=0)
    axes[i].plot(start_points/fs, corr_match_avg[:,0], label='TASK 1')
    axes[i].plot(start_points/fs, corr_match_avg[:,1], label='TASK 2')
    axes[i].plot(start_points/fs, corr_match_avg[:,2], label='TASK 3')
    axes[i].set_title(f'Subject {i+1}')
    if i == 0:
        axes[i].legend()

In [ ]:
# plot for all videos, with marker info
import ast
eeg_files_all = [file for file in os.listdir('../../Experiments/data/SOMove_MultiTask/Subj_1') if file.endswith('.set')]
files = [file for file in eeg_files_all if file[-5] == '1']
files.sort()
video_ids = [int(file.split('_')[0]) for file in files]
marker_info_task_12 = pd.read_csv('../../Experiments/data/SOMove_MultiTask/mounted_videos_info.csv')
marker_info_task_12 = marker_info_task_12[marker_info_task_12['videoID'].isin(video_ids)]
cross_task12 = marker_info_task_12['frames_cross'].values.tolist()
cross_task12 = [ast.literal_eval(cross) for cross in cross_task12]
circle_task12 = marker_info_task_12['frames_circle'].values.tolist()
circle_task12 = [ast.literal_eval(circle) for circle in circle_task12]
marker_info_task_3 = pd.read_csv('../../Experiments/data/SOMove_MultiTask/mounted_videos_info_3.csv')
marker_info_task_3 = marker_info_task_3[marker_info_task_3['videoID'].isin(video_ids)]
cross_task3 = marker_info_task_3['frames_cross'].values.tolist()
cross_task3 = [ast.literal_eval(cross) for cross in cross_task3]
circle_task3 = marker_info_task_3['frames_circle'].values.tolist()
circle_task3 = [ast.literal_eval(circle) for circle in circle_task3]
TASK = 2
if TASK != 3:
    cross_task = cross_task12
    circle_task = circle_task12
else:
    cross_task = cross_task3
    circle_task = circle_task3
plt.close('all')
corr_match_folds = np.split(corr_match_avg_subj, 7)
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharex=True, sharey=True)
axes = axes.flatten()
for i, corr_match in enumerate(corr_match_folds):
    cross = cross_task[i] 
    circle = circle_task[i] 
    axes[i].plot(start_points/fs, corr_match[:,TASK-1])
    axes[i].set_title(f'Video {i+1}')
    # plot the markers
    for j in range(len(cross)):
        axes[i].axvline(x=cross[j]/fs-1, color='r', linestyle='--', alpha=0.5)
    for j in range(len(circle)):
        axes[i].axvline(x=circle[j]/fs-1, color='g', linestyle='--', alpha=0.5)
    

In [ ]:
# plot for all videos
plt.close('all')
corr_match_folds = np.split(corr_match_avg_subj, 7)
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharex=True, sharey=True)
axes = axes.flatten()
for i, corr_match in enumerate(corr_match_folds):
    axes[i].plot(start_points/fs, corr_match[:,0], label='TASK 1')
    axes[i].plot(start_points/fs, corr_match[:,1], label='TASK 2')
    axes[i].plot(start_points/fs, corr_match[:,2], label='TASK 3')
    axes[i].set_title(f'Video {i+1}')
    if i == 0:
        axes[i].legend()

In [ ]:
plt.close('all')
task = 3
corr_match_folds = np.split(corr_match_avg_subj, 7)
corr_match_avg = np.mean(corr_match_folds, axis=0)
corr_mismatch_folds = np.split(corr_mismatch_avg_subj, 7)
corr_mismatch_avg = np.mean(corr_mismatch_folds, axis=0)
plt.scatter(start_points/fs, corr_match_avg[:,task-1], label='Match')
plt.scatter(start_points/fs, corr_mismatch_avg[:,task-1], label='Mismatch')
plt.show()